### Allscripts Sunrise (SCM) Visit Detail Hydration

Visit detail rows are derived from `dbo_cv3clientvisit` and linked back to the already-populated Sunrise visit backbone.

In [0]:
# %python
# required_columns = [
#     ('person_id', 'BIGINT'),
#     ('provider_id', 'BIGINT'),
#     ('care_site_id', 'BIGINT'),
#     ('preceding_visit_detail_id', 'BIGINT'),
#     ('parent_visit_detail_id', 'BIGINT'),
#     ('visit_occurrence_id', 'BIGINT')
# ]
# existing_columns = {field.name for field in spark.table('_exponent.omop_silver.visit_detail').schema.fields}
# missing_columns = [f"{name} {definition}" for name, definition in required_columns if name not in existing_columns]
# if missing_columns:
#     spark.sql('ALTER TABLE _exponent.omop_silver.visit_detail ADD COLUMNS (' + ', '.join(missing_columns) + ')')
# print(missing_columns if missing_columns else 'No missing columns')

In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.visit_detail;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.visit_detail
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_visit_detail
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_visit_detail AS
SELECT
  stp.person_id,
  CASE
    WHEN UPPER(v.TypeCode) = 'INPATIENT' THEN 9201
    WHEN UPPER(v.TypeCode) IN ('AMBULATORY', 'OUTPATIENT') THEN 9202
    WHEN UPPER(v.TypeCode) = 'EMERGENCY' THEN 9203
    ELSE 0
  END AS visit_detail_concept_id,
  CAST(v.AdmitDtm AS DATE) AS visit_detail_start_date,
  v.AdmitDtm AS visit_detail_start_datetime,
  CAST(COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm) AS DATE) AS visit_detail_end_date,
  COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm) AS visit_detail_end_datetime,
  32817 AS visit_detail_type_concept_id,
  NULL AS provider_id,
  NULL AS care_site_id,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(v.GUID AS STRING)) AS visit_detail_source_value,
  0 AS visit_detail_source_concept_id,
  0 AS admitted_from_concept_id,
  NULL AS admitted_from_source_value,
  0 AS discharged_to_concept_id,
  v.DischargeDisposition AS discharged_to_source_value,
  NULL AS preceding_visit_detail_id,
  NULL AS parent_visit_detail_id,
  stvo.visit_occurrence_id AS visit_occurrence_id,
  'allscripts_scm' AS source_system,
  CURRENT_TIMESTAMP() AS last_mod_tsp
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit v
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(v.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON stvo.visit_occurrence_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(v.GUID AS STRING))
 AND stvo.active_flag = TRUE
WHERE 1=1
  AND v.visitstatus = 'CAN'
  AND v.GUID IS NOT NULL
  AND v.ClientGUID IS NOT NULL
  AND v.AdmitDtm IS NOT NULL
  AND v.AdmitDtm >= TIMESTAMP('1900-01-01')
  AND v.AdmitDtm <= CURRENT_TIMESTAMP()
  AND COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm) >= TIMESTAMP('1900-01-01')
  AND COALESCE(v.DischargeDtm, v.CloseDtm, v.AdmitDtm) <= CURRENT_TIMESTAMP();

In [0]:
%sql
MERGE INTO _exponent.omop_silver.visit_detail AS t
USING silver_visit_detail AS s
ON t.visit_detail_source_value = s.visit_detail_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.visit_detail_concept_id <=> s.visit_detail_concept_id)
  OR NOT (t.visit_detail_start_date <=> s.visit_detail_start_date)
  OR NOT (t.visit_detail_start_datetime <=> s.visit_detail_start_datetime)
  OR NOT (t.visit_detail_end_date <=> s.visit_detail_end_date)
  OR NOT (t.visit_detail_end_datetime <=> s.visit_detail_end_datetime)
  OR NOT (t.visit_detail_type_concept_id <=> s.visit_detail_type_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id = s.person_id,
  t.visit_detail_concept_id = s.visit_detail_concept_id,
  t.visit_detail_start_date = s.visit_detail_start_date,
  t.visit_detail_start_datetime = s.visit_detail_start_datetime,
  t.visit_detail_end_date = s.visit_detail_end_date,
  t.visit_detail_end_datetime = s.visit_detail_end_datetime,
  t.visit_detail_type_concept_id = s.visit_detail_type_concept_id,
  t.provider_id = s.provider_id,
  t.care_site_id = s.care_site_id,
  t.visit_detail_source_concept_id = s.visit_detail_source_concept_id,
  t.admitted_from_concept_id = s.admitted_from_concept_id,
  t.admitted_from_source_value = s.admitted_from_source_value,
  t.discharged_to_concept_id = s.discharged_to_concept_id,
  t.discharged_to_source_value = s.discharged_to_source_value,
  t.preceding_visit_detail_id = s.preceding_visit_detail_id,
  t.parent_visit_detail_id = s.parent_visit_detail_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.source_system = s.source_system,
  t.last_mod_tsp = s.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.person_id,
  s.visit_detail_concept_id,
  s.visit_detail_start_date,
  s.visit_detail_start_datetime,
  s.visit_detail_end_date,
  s.visit_detail_end_datetime,
  s.visit_detail_type_concept_id,
  s.provider_id,
  s.care_site_id,
  s.visit_detail_source_value,
  s.visit_detail_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.preceding_visit_detail_id,
  s.parent_visit_detail_id,
  s.visit_occurrence_id,
  s.source_system,
  s.last_mod_tsp
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_detail (
  source_system,
  visit_detail_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.visit_detail_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM _exponent.omop_silver.visit_detail s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_detail x
  ON s.visit_detail_source_value = x.visit_detail_source_value
WHERE s.source_system = 'allscripts_scm';

In [0]:
%sql
MERGE INTO _exponent.omop_scm.visit_detail AS gold
USING (
  SELECT
    svd.visit_detail_id,
    s.person_id,
    s.visit_detail_concept_id,
    s.visit_detail_start_date,
    s.visit_detail_start_datetime,
    s.visit_detail_end_date,
    s.visit_detail_end_datetime,
    s.visit_detail_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_detail_source_value,
    s.visit_detail_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_detail_id,
    s.parent_visit_detail_id,
    s.visit_occurrence_id
  FROM _exponent.omop_silver.visit_detail s
  JOIN _exponent.omop_mapping.source_to_visit_detail svd
    ON svd.visit_detail_source_value = s.visit_detail_source_value
   AND svd.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
    AND s.visit_occurrence_id IS NOT NULL
) AS src
ON gold.visit_detail_id = src.visit_detail_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.visit_detail_concept_id = src.visit_detail_concept_id,
  gold.visit_detail_start_date = src.visit_detail_start_date,
  gold.visit_detail_start_datetime = src.visit_detail_start_datetime,
  gold.visit_detail_end_date = src.visit_detail_end_date,
  gold.visit_detail_end_datetime = src.visit_detail_end_datetime,
  gold.visit_detail_type_concept_id = src.visit_detail_type_concept_id,
  gold.provider_id = src.provider_id,
  gold.care_site_id = src.care_site_id,
  gold.visit_detail_source_value = src.visit_detail_source_value,
  gold.visit_detail_source_concept_id = src.visit_detail_source_concept_id,
  gold.admitted_from_concept_id = src.admitted_from_concept_id,
  gold.admitted_from_source_value = src.admitted_from_source_value,
  gold.discharged_to_concept_id = src.discharged_to_concept_id,
  gold.discharged_to_source_value = src.discharged_to_source_value,
  gold.preceding_visit_detail_id = src.preceding_visit_detail_id,
  gold.parent_visit_detail_id = src.parent_visit_detail_id,
  gold.visit_occurrence_id = src.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_detail_id,
  person_id,
  visit_detail_concept_id,
  visit_detail_start_date,
  visit_detail_start_datetime,
  visit_detail_end_date,
  visit_detail_end_datetime,
  visit_detail_type_concept_id,
  provider_id,
  care_site_id,
  visit_detail_source_value,
  visit_detail_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_detail_id,
  parent_visit_detail_id,
  visit_occurrence_id
)
VALUES (
  src.visit_detail_id,
  src.person_id,
  src.visit_detail_concept_id,
  src.visit_detail_start_date,
  src.visit_detail_start_datetime,
  src.visit_detail_end_date,
  src.visit_detail_end_datetime,
  src.visit_detail_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_detail_source_value,
  src.visit_detail_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_detail_id,
  src.parent_visit_detail_id,
  src.visit_occurrence_id
);